#### Builtin methods defined on **`Stream`** / **`Trace`**

- Most methods that work on a Trace object also work on a Stream object. They are simply executed for every trace. [See ObsPy documentation for an overview of available methods](http://docs.obspy.org/packages/autogen/obspy.core.stream.Stream.html) (or try **`st.<Tab>`**).
 - **`st.filter()`** - Filter all attached traces.
 - **`st.trim()`** - Cut all traces.
 - **`st.resample()`** / **`st.decimate()`** - Change the sampling rate.
 - **`st.trigger()`** - Run triggering algorithms.
 - **`st.plot()`** / **`st.spectrogram()`** - Visualize the data.
 - **`st.attach_response()`**/**`st.remove_response()`**, **`st.simulate()`** - Instrument correction
 - **`st.merge()`**, **`st.normalize()`**, **`st.detrend()`**, **`st.taper()`**, ...
- A **`Stream`** object can also be exported to many formats, so ObsPy can be used to convert between different file formats.

#### Trace Exercise
 - Make an **`numpy.ndarray`** with zeros and (e.g. use **`numpy.zeros()`**) and put an ideal pulse somewhere in it
 - initialize a **`Trace`** object with your data array
 - Fill in some station information (e.g. network, station, ..)
 - Print trace summary and plot the trace
 - Change the sampling rate to 20 Hz
 - Change the starttime of the trace to the start time of this session
 - Print the trace summary and plot the trace again
- Use **`tr.filter(...)`** and apply a lowpass filter with a corner frequency of 1 Hertz.
- Display the preview plot, there are a few seconds of zeros that we can cut off.
- Scale up the amplitudes of the trace by a factor of 500
- Add standard normal gaussian noise to the trace (use [**`np.random.randn()`**](http://docs.scipy.org/doc/numpy/reference/generated/numpy.random.randn.html))
- Display the preview plot again


# Local files
Information about station II.PFO:
https://www.fdsn.org/station_book/II/PFO/pfo_14.html

In [ ]:
from obspy import read, read_inventory
st = read("./data/waveform_PFO.mseed")
print(st)
st.resample(sampling_rate=20.0)
st.detrend("linear")
st.taper(max_percentage=0.05, type='cosine')
#st.filter("lowpass", freq=0.1)
#st.plot();
st.filter("bandpass", freqmin=0.01, freqmax=0.1).plot();



Let's cross correlate the signals from these co-located vertical sensors:

In [ ]:

from obspy.signal.cross_correlation import correlate, xcorr_max
import matplotlib.pyplot as plt

def plot_xcorr(st_in):
    st = st_in.copy()
    fs = min([tr.stats.sampling_rate for tr in st])
    st.resample(sampling_rate=fs)
    cc = correlate(st[0], st[1], int(st[0].stats.sampling_rate))

    plt.figure()
    plt.plot(cc)
    plt.xlabel('Shift (samples)')
    plt.ylabel('Cross correlation coefficient')

    shift, value = xcorr_max(cc)
    print(f'max xcorr is {value} at a shift of {shift} samples')

plot_xcorr(st)

# numpy array properties
* size
* shape

# numpy array methods
* min
* max
* abs
* mean
* std
* sum
* cumsum
* prod
* cumprod
* clip
* round
* sort



In [ ]:
tr = st[0].copy()
tr.data = tr.data[10000:11000:50].round()
tr.plot();
print(f'data={tr.data}')
print(f'size={tr.data.size}')
print(f'shape={tr.data.shape}')
print(f'sum={tr.data.sum()}')
print(f'cumulative sum={tr.data.cumsum()}')
print(f'mean={tr.data.mean()}')
print(f'max={tr.data.max()}')
print(f'absmax={np.abs(tr.data).max()}')
print(f'peak to peak={tr.data.ptp()}')

# clipping
tr2 = tr.copy()
tr2.data=tr2.data.clip(-100, 100)
tr2.plot();

print('sorting')
tr.data.sort()
print(f'sorted data={tr.data}')
tr.plot();




In [ ]:
from obspy import read_inventory
inv = read_inventory("./data/station_PFO.xml", format="STATIONXML")
#inv.get_response(st[0].id, st[0].stats.starttime).plot(0.01);
print(type(inventory))

st = read("./data/waveform_PFO.mseed")
st.remove_response(inventory=inv)
st.plot();
plot_xcorr(st)

st = read("./data/waveform_PFO.mseed")
st.remove_response(inventory=inv, water_level=60, pre_filt=(0.01, 0.02, 8, 10), output="DISP")
st.plot();
plot_xcorr(st)

In [ ]:
from obspy import read_events

catalog = read_events("./data/event_tohoku_with_big_aftershocks.xml")
print(catalog)

### Make your own

In [ ]:
from obspy import Stream, Trace, UTCDateTime

x = np.random.randint(-100, 100, 500)
tr = Trace(data=x)
tr.stats.station = "XYZ"
tr.stats.starttime = UTCDateTime()

tr2 = Trace(data=np.random.randint(-300, 100, 1000))
tr2.stats.starttime = UTCDateTime()
tr2.stats.sampling_rate = 10.0
st = Stream([tr, tr2])

print(st)
st.plot();



In [ ]:
from obspy import UTCDateTime
from obspy.core.event import Catalog, Event, Origin, Magnitude
from obspy.geodetics import FlinnEngdahl

cat = Catalog()
cat.description = "Just a fictitious toy example catalog built from scratch"

e = Event()
e.event_type = "not existing"

o = Origin()
o.time = UTCDateTime(2014, 2, 23, 18, 0, 0)
o.latitude = 47.6
o.longitude = 12.0
o.depth = 10000
o.depth_type = "operator assigned"
o.evaluation_mode = "manual"
o.evaluation_status = "preliminary"
o.region = FlinnEngdahl().get_region(o.longitude, o.latitude)

m = Magnitude()
m.mag = 7.2
m.magnitude_type = "Mw"

m2 = Magnitude()
m2.mag = 7.4
m2.magnitude_type = "Ms"

# also included could be: custom picks, amplitude measurements, station magnitudes,
# focal mechanisms, moment tensors, ...

# make associations, put everything together
cat.append(e)
e.origins = [o]
e.magnitudes = [m, m2]
m.origin_id = o.resource_id
m2.origin_id = o.resource_id

print(cat)
cat.write("/tmp/my_custom_events.xml", format="QUAKEML")
!cat /tmp/my_custom_events.xml

### FDSN

ObsPy has clients to directly fetch data via...

- FDSN webservices (IRIS, Geofon/GFZ, USGS, NCEDC, SeisComp3 instances, ...)
- ArcLink (EIDA, ...)
- Earthworm
- SeedLink (near-realtime servers)
- NERIES/NERA/seismicportal.eu
- NEIC
- SeisHub (local seismological database)

This introduction shows how to use the FDSN webservice client. The FDSN webservice definition is by now the default web service implemented by many data centers world wide. Clients for other protocols work similar to the FDSN client.

#### Waveform Data

In [ ]:
from obspy import UTCDateTime
from obspy.clients.fdsn import Client

client = Client("IRIS")
t = UTCDateTime("2011-03-11T05:46:23")  # Tohoku
st = client.get_waveforms("II", "PFO", "*", "LHZ",
                          t + 10 * 60, t + 30 * 60)
print(st)
st.plot();

In [ ]:
import obspy
from obspy.clients.fdsn import Client

c_event = Client("USGS")

# Event time.
event_time = obspy.UTCDateTime("2011-03-11T05:46:23.2")

# Get the event information. The temporal and magnitude constraints make it unique
cat = c_event.get_events(starttime=event_time - 10, endtime=event_time + 10,
                         minmagnitude=9)
print(cat)

c = Client("IRIS")
# Download station information at the response level!
inv = c.get_stations(network="II", station="BFO", location="*", channel="BH?",
                     starttime=event_time - 60, endtime=event_time + 3600,
                     level="response")
print(inv)

# Download 3 component waveforms.
st = c.get_waveforms(network="II", station="BFO", location="*",
                     channel="BH?", starttime=event_time - 60,
                     endtime=event_time + 3600)
print(st)